In [1]:
def mix(p_step, p_secret_number):
    return p_step ^ p_secret_number

# # Test (If the secret number is 42 and you were to mix 15 into the secret number, the secret number would become 37.

# mix(42, 15)

In [2]:
def prune(p_secret_number):
    return p_secret_number % 16777216

# # Test
# 16113920
# prune(100_000_000)

In [3]:
def next_secret_number(p_secret_number):
    secret_number = p_secret_number
    result = secret_number * 64
    secret_number = mix(result, secret_number)
    secret_number = prune(secret_number)
    result = secret_number // 32
    secret_number = mix(result, secret_number)
    secret_number = prune(secret_number)
    result = secret_number * 2048
    secret_number = mix(result, secret_number)
    secret_number = prune(secret_number)  
    return secret_number

# # Test
# next_secret_number(123)

In [4]:
def calc_n_secrets(p_secret_number, cycles, debug=False):
    new_secret_number = p_secret_number
    for i in range(cycles):
        new_secret_number = next_secret_number(new_secret_number)
        print(f"{i+1}: {new_secret_number}") if debug else None
    return new_secret_number

# # Test
# # So, if a buyer had a secret number of 123, that buyer's next ten secret numbers would be:

# # 15887950
# # 16495136
# # 527345
# # 704524
# # 1553684
# # 12683156
# # 11100544
# # 12249484
# # 7753432
# # 5908254


# calc_n_secrets(123, 10, True)
# print('-'*40)
# s = 1 
# print(f'{s}: {calc_n_secrets(s, 2000)}')
# s = 10 
# print(f'{s}: {calc_n_secrets(s, 2000)}')
# s = 100
# print(f'{s}: {calc_n_secrets(s, 2000)}')
# s = 2024
# print(f'{s}: {calc_n_secrets(s, 2000)}')


In [5]:
l_input = [1, 10, 100, 2024]

In [7]:
def calc_list(p_input):
    acc = 0
    for s in p_input:
       acc += calc_n_secrets(s, 2000) 
    return acc

# # Test
# calc_list(l_input)

In [10]:
def take_input(file_name):
    l_input = []
    with open(file_name, 'r') as file:
        for row in file:
            l_input.append(int(row.strip()))
    return l_input

# # Test
# l_input = take_input('input.txt')
# l_input

In [11]:
l_input = take_input('input.txt')
sum_2000 = calc_list(l_input)
sum_2000

19927218456

In [75]:
counter_template = {
    'position': 'A',
    'elements': 0,
    'A': None,
    'B': None,
    'C': None,
    'D': None,
    'E': None
    }

next_counter_position = {
    'A': 'B',
    'B': 'C',
    'C': 'D',
    'D': 'E',
    'E': 'A'
}


buyers_sequences_template = {
    00000000: {5808198: 1, 
               14138889: 2} 
}

buyers_sequences = {}

# Test

print (counter_template)

{'position': 'A', 'elements': 0, 'A': None, 'B': None, 'C': None, 'D': None, 'E': None}


In [65]:
def calc_sequence(p_counter, debug=False):
    position_01 = next_counter_position[p_counter['position']]
    position_02 = next_counter_position[position_01]
    position_03 = next_counter_position[position_02]
    position_04 = next_counter_position[position_03]
    position_05 = next_counter_position[position_04]

    diff_01 = p_counter[position_02] - p_counter[position_01]
    diff_02 = p_counter[position_03] - p_counter[position_02]
    diff_03 = p_counter[position_04] - p_counter[position_03]
    diff_04 = p_counter[position_05] - p_counter[position_04]  

    print(f'diff_01 = {diff_01}, diff_02 = {diff_02}, diff_03 = {diff_03}, diff_04 = {diff_04}') if debug else None

    acc = 0 
    
    for diff in (diff_01, diff_02, diff_03, diff_04):
        if diff < 0:
            acc = acc * 100 + (10 + abs(diff))
        else:
            acc = (acc * 100) + diff      
           
        print(f'diff = {diff}, acc = {acc}') if debug else None

    return acc

# # Test

# p_counter = counter_template.copy()

# p_counter['A'] = 3
# p_counter['B'] = 0
# p_counter['C'] = 6
# p_counter['D'] = 5
# p_counter['E'] = 4    
# p_counter['elements'] = 5
# p_counter['position'] = 'E'

# #calc_sequence(p_counter)
# print (p_counter)
# print (calc_sequence(p_counter, False))

In [32]:
def get_current_prise(p_counter):
    position = p_counter['position']
    return p_counter[position]

# # Test

# p_counter = counter_template

# p_counter['^'] = 0
# p_counter['>'] = 3
# p_counter['v'] = 6
# p_counter['<'] = 9
# p_counter['elements'] = 4
# p_counter['position'] = '^'

# get_current_prise(p_counter)

In [74]:
def register_sequence(p_secret_number, p_counter):
    sequence = calc_sequence(p_counter)
    price = get_current_prise(p_counter)
    if not sequence in buyers_sequences:
        buyers_sequences[sequence] = {p_secret_number: price}
    elif p_secret_number not in buyers_sequences[sequence]:
        buyers_sequences[sequence][p_secret_number] = price
    else:
        None # There is already previous price for this sequence set

# # Test

# p_counter = counter_template.copy()

# p_counter['A'] = 3
# p_counter['B'] = 0
# p_counter['C'] = 6
# p_counter['D'] = 5
# p_counter['E'] = 4    
# p_counter['elements'] = 5
# p_counter['position'] = 'B'

# register_sequence(123, p_counter)

In [76]:
buyers_sequences

{}

In [80]:
def calc_price_changes(p_secret_number, cycles, debug=False):
    new_secret_number = p_secret_number
    counter = counter_template
    counter['elements'] = 1
    counter['A'] = new_secret_number
    for i in range(cycles):
        new_secret_number = next_secret_number(new_secret_number)
        print(f"{i+1}: {new_secret_number}") if debug else None  

        counter['position'] = next_counter_position[counter['position']] # Chnge to the next position
        counter[counter['position']] = new_secret_number % 10
        if counter['elements'] < 5:
            counter['elements'] += 1
        else:
            register_sequence(p_secret_number, counter)
    
    return new_secret_number

# Test
calc_price_changes(1, 1, True)

1: 137283


137283

In [77]:
def calc_buyers_sequences(p_input):
    for s in p_input:
       calc_price_changes(s, 2000) 



In [81]:
l_input = take_input('input.txt')
calc_buyers_sequences(l_input)
buyers_sequences

{6111502: {5808198: 4,
  14046433: 4,
  10358780: 2,
  7361422: 3,
  13352583: 2,
  1093182: 3,
  8448264: 3,
  14588831: 5,
  11131029: 2,
  5769457: 2,
  4765107: 2,
  5380340: 5,
  1360428: 2,
  11484233: 2,
  12049221: 3,
  12126347: 5,
  9796528: 2,
  482089: 5,
  14545974: 3,
  9983865: 2,
  13125663: 2,
  15773660: 5,
  12224452: 3,
  4197580: 5,
  15876544: 5,
  1912384: 3,
  4349181: 3,
  5790078: 5,
  12051087: 4,
  11115664: 5,
  7104816: 3,
  2195367: 3,
  15848881: 5,
  6576168: 4,
  13383999: 5,
  16542783: 4,
  10897401: 4,
  10746160: 4,
  2497530: 3,
  8144184: 4,
  1954555: 2,
  13718243: 4,
  10260594: 2,
  9413188: 5,
  5539637: 2,
  4875840: 3,
  3756194: 2,
  10650896: 2,
  7705249: 3,
  1577942: 4,
  381282: 5,
  8412188: 3,
  14665864: 2,
  14644317: 2,
  8703583: 5,
  8640715: 2,
  7377673: 2,
  3939240: 2,
  951465: 5,
  4728793: 5,
  1932379: 5,
  5943279: 4,
  15774883: 2,
  1950405: 3,
  8094840: 3,
  464712: 4,
  2215596: 5,
  4079101: 3,
  1550087: 3,
  2

In [82]:
def calc_best_sequence(p_buyers_sequences):
    best_seq = 0
    best_price = 0
    for seq in p_buyers_sequences:
        seq_price = 0
        for buyer in p_buyers_sequences[seq]:
            seq_price += p_buyers_sequences[seq][buyer]
        if seq_price > best_price:
            best_price = seq_price
            best_seq = seq

    return (best_seq, best_price)

# Test

print(calc_best_sequence(buyers_sequences))


(11020000, 2189)
